# Eval Viewer
Interactive notebook to visualize per-video predictions from a trained model.

1. Set `RESULTS_DIR` to a completed training run (the folder containing `config.yml` and fold subfolders).
2. Run all cells.
3. Use the slider to browse videos and see segment predictions + probability curves.

In [1]:
# ── Config ──────────────────────────────────────────────
RESULTS_DIR = "/code/jjiang23/BalanceTestThesis/results/MAMP/MB/downsamp/ASFormer/20260603_224342"
DEVICE = "cuda"        # or "cpu"
STRIDE_OVERRIDE = 15   # smaller stride = smoother stitching (None = use data config stride)

In [2]:
import os, sys, json, yaml
import importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(RESULTS_DIR), "..", "..", "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.eval.metric_utils import predict_video, compute_segmentation_metrics, extract_segments

print(f"Project root: {PROJECT_ROOT}")

Project root: /code/jjiang23/BalanceTestThesis


In [3]:
# ── Load config ─────────────────────────────────────────
config_path = os.path.join(RESULTS_DIR, "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

e_cfg = config["encoder"]
d_cfg = config["data"]
t_cfg = config["trainer"]
s_cfg = config["segmentor"]
splits_path = config["paths"]["splits_path"]

with open(splits_path) as f:
    splits = json.load(f)

print(f"Loaded config from: {config_path}")
print(f"Folds: {list(splits.keys())}")

Loaded config from: /code/jjiang23/BalanceTestThesis/results/MAMP/MB/downsamp/ASFormer/20260603_224342/config.yml
Folds: ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']


In [4]:
# ── Dynamic import helper ───────────────────────────────
def load_module_from_path(file_path):
    spec = importlib.util.spec_from_file_location("mod", file_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# ── Load initializers ───────────────────────────────────
if e_cfg is not None:
    init_encoder_path = os.path.abspath(e_cfg["init_encoder_path"])
else:
    init_encoder_path = os.path.join(PROJECT_ROOT, "initializers", "encoder", "Identity.py")

init_segmentor_path = os.path.abspath(s_cfg["init_segmentor_path"])

initialize_encoder = getattr(load_module_from_path(init_encoder_path), "initialize_encoder")
initialize_segmentor = getattr(load_module_from_path(init_segmentor_path), "initialize_segmentor")

# ── Build models (architecture only — weights loaded per fold) ──
encoder = initialize_encoder(d_cfg, e_cfg)
segmentor = initialize_segmentor(
    s_cfg, encoder,
    class_weights=None,
    lambda_smooth=t_cfg.get("lambda_smooth", 0.01),
    time_alignment=t_cfg.get("time_alignment", "downsample_labels"),
)
encoder.to(DEVICE).eval()
segmentor.to(DEVICE).eval()
print("Models ready.")

num_joints 25 patch_size 1 num_frames 120 t_patch_size 4


/code/jjiang23/BalanceTestThesis/models/encoders/MY_MAMP/encoder.py:376: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=

Loaded MAMP checkpoint from: /data2/pathml/MAMP/checkpoints/checkpoints/ntu120_xset.pth
Model class: model_mamp.transformer.Transformer
temporal_patch_size: 4
out_dim: 256
✓ MAMP encoder loaded successfully
  Config: model_mamp.transformer.Transformer
  Model parameters: 10368524
✓ ASFormer segmentor initialized (in_dim=256, classes=2, num_f_maps=64, layers=10, decoders=3, r1=2, r2=2, att_type=block_att, channel_masking_rate=0.3, lambda_smooth=0.01, alignment=downsample_labels)
  Parameters: 1,012,104 total, 1,012,104 trainable
  Class weights: none (uniform)
Models ready.


In [5]:
# ── Collect all (fold, video) pairs ─────────────────────
video_entries = []  # list of (fold_name, video_path, fold_dir)

for fold_name, split_files in splits.items():
    fold_dir = os.path.join(RESULTS_DIR, fold_name)
    enc_ckpt = os.path.join(fold_dir, "best_encoder.pt")
    seg_ckpt = os.path.join(fold_dir, "best_segmentor.pt")
    if not (os.path.exists(enc_ckpt) and os.path.exists(seg_ckpt)):
        print(f"Skipping {fold_name}: no checkpoints")
        continue
    for vid_path in split_files["val"]:
        video_entries.append((fold_name, vid_path, fold_dir))

print(f"Total eval videos: {len(video_entries)} across {len(splits)} folds")

Total eval videos: 128 across 5 folds


In [6]:
# ── Cache for loaded fold weights ──────────────────────
_loaded_fold = None

def load_fold_weights(fold_dir):
    """Load encoder + segmentor weights for a fold (cached)."""
    global _loaded_fold
    if _loaded_fold == fold_dir:
        return
    encoder.load_state_dict(
        torch.load(os.path.join(fold_dir, "best_encoder.pt"), map_location=DEVICE)
    )
    segmentor.load_state_dict(
        torch.load(os.path.join(fold_dir, "best_segmentor.pt"), map_location=DEVICE),
        strict=False,
    )
    encoder.eval()
    segmentor.eval()
    _loaded_fold = fold_dir

In [ ]:
# ── Video frame utilities ──────────────────────────
import h5py
import cv2

def get_video_meta_from_h5(h5_path):
    """
    Returns (video_path, box_coords) from h5 file.
    box_coords: (x1, y1, x2, y2) in pixel space, or None if not present.
    """
    try:
        with h5py.File(h5_path, 'r') as f:
            vp = f.attrs.get('video_path', None)
            if vp is not None:
                vp = vp.decode() if isinstance(vp, bytes) else str(vp)
            box = None
            if 'static_box_coords' in f:
                box = f['static_box_coords'][:].tolist()
        return vp, box
    except Exception as e:
        print(f'Warning: could not read meta from {h5_path}: {e}')
        return None, None


def read_video_frame(video_path, frame_idx):
    """Read a single RGB frame from a video file using OpenCV."""
    if video_path is None:
        return None
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f'Cannot open video: {video_path}')
            return None
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            return None
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f'Warning: frame {frame_idx} read failed: {e}')
        return None


print('Video frame utilities ready.')


In [ ]:
# ── Visualization helpers + prediction cache ─────────────
import matplotlib.patches as mpatches

CLASS_COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#FFC107', '#9C27B0']
CLASS_NAMES  = ['Phase', 'Nonphase']

# ── Global prediction cache (reused by frame slider) ──────
_pred_cache = {
    'vid_idx': None, 'gt': None, 'pred': None, 'probs': None,
    'video_path': None, 'box_coords': None,
    'fold_name': None, 'vid_name': None,
    'metrics': None, 'frame_acc': None,
}


def _ensure_predictions(vid_idx):
    """Run inference for vid_idx only if not already cached."""
    if _pred_cache['vid_idx'] == vid_idx:
        return
    fold_name, vid_path, fold_dir = video_entries[vid_idx]
    load_fold_weights(fold_dir)
    gt, pred, probs = predict_video(
        vid_path, encoder, segmentor, d_cfg, DEVICE,
        stride_override=STRIDE_OVERRIDE,
    )
    valid = gt != -100
    metrics = compute_segmentation_metrics(
        gt, pred, class_id=0, iou_thresholds=(0.1, 0.25, 0.5), fps=30,
    )
    video_path, box_coords = get_video_meta_from_h5(vid_path)
    _pred_cache.update({
        'vid_idx':    vid_idx,
        'gt':         gt,
        'pred':       pred,
        'probs':      probs,
        'video_path': video_path,
        'box_coords': box_coords,
        'fold_name':  fold_name,
        'vid_name':   os.path.basename(vid_path),
        'metrics':    metrics,
        'frame_acc':  float((gt[valid] == pred[valid]).mean()) if valid.sum() > 0 else 0.0,
    })


def _draw_chart(frame_cursor=None):
    """Draw the 3-row GT/Pred/Prob chart with an optional cursor line."""
    gt    = _pred_cache['gt']
    pred  = _pred_cache['pred']
    probs = _pred_cache['probs']
    T = len(gt)
    frames = np.arange(T)
    num_classes = probs.shape[1]
    m = _pred_cache['metrics']
    f1_10  = m.get('f1_iou_0.1',  0) or 0
    f1_25  = m.get('f1_iou_0.25', 0) or 0
    f1_50  = m.get('f1_iou_0.5',  0) or 0

    fig, axes = plt.subplots(
        3, 1, figsize=(18, 7), sharex=True,
        gridspec_kw={'height_ratios': [1, 1, 2.5], 'hspace': 0.08},
    )
    ax_gt, ax_pred, ax_prob = axes

    for c in range(num_classes):
        color = CLASS_COLORS[c % len(CLASS_COLORS)]
        name  = CLASS_NAMES[c] if c < len(CLASS_NAMES) else f'Class {c}'
        ax_gt.fill_between(frames, 0, 1, where=(gt == c),
                            color=color, alpha=0.9, label=name)
        ax_pred.fill_between(frames, 0, 1, where=(pred == c),
                              color=color, alpha=0.9)
        ax_prob.plot(frames, probs[:, c], label=name,
                     color=color, lw=1.2)

    ax_gt.set_yticks([]);   ax_gt.set_ylabel('GT',   fontsize=10, fontweight='bold')
    ax_pred.set_yticks([]); ax_pred.set_ylabel('Pred', fontsize=10, fontweight='bold')
    ax_gt.legend(loc='upper right', fontsize=7, ncol=num_classes)
    ax_prob.set_ylim(-0.05, 1.05)
    ax_prob.set_ylabel('Probability', fontsize=10)
    ax_prob.set_xlabel('Frame', fontsize=10)
    ax_prob.legend(loc='upper right', fontsize=7)
    ax_prob.grid(axis='y', alpha=0.3)

    if frame_cursor is not None:
        for ax in axes:
            ax.axvline(x=frame_cursor, color='white', lw=1.8, alpha=0.9, zorder=5)

    vid_name  = _pred_cache['vid_name']
    fold_name = _pred_cache['fold_name']
    frame_acc = _pred_cache['frame_acc']
    fig.suptitle(
        f'[{fold_name}] {vid_name}   |   Acc: {frame_acc:.1%}   '
        f'F1@.10: {f1_10:.2f}  F1@.25: {f1_25:.2f}  F1@.50: {f1_50:.2f}   ({T} frames)',
        fontsize=11, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    plt.show()


def _draw_frame(frame_idx):
    """Display the video frame at frame_idx with GT/pred label and bbox overlay."""
    vp  = _pred_cache['video_path']
    box = _pred_cache['box_coords']   # (x1, y1, x2, y2) or None

    fig, ax = plt.subplots(figsize=(6, 5))
    frame_img = read_video_frame(vp, frame_idx) if vp else None

    if frame_img is not None:
        ax.imshow(frame_img)

        # Draw static bounding box
        if box is not None and len(box) == 4:
            x1, y1, x2, y2 = box
            rect = mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor='lime', facecolor='none',
                label=f'box ({x1:.0f},{y1:.0f})→({x2:.0f},{y2:.0f})',
            )
            ax.add_patch(rect)
            ax.legend(loc='lower right', fontsize=7, framealpha=0.7)

        gt_lbl   = int(_pred_cache['gt'][frame_idx])
        pred_lbl = int(_pred_cache['pred'][frame_idx])
        gt_name   = CLASS_NAMES[gt_lbl]   if 0 <= gt_lbl   < len(CLASS_NAMES) else str(gt_lbl)
        pred_name = CLASS_NAMES[pred_lbl] if 0 <= pred_lbl < len(CLASS_NAMES) else str(pred_lbl)
        correct   = '\u2713' if gt_lbl == pred_lbl else '\u2717'
        color     = 'green' if gt_lbl == pred_lbl else 'red'
        ax.set_title(
            f'Frame {frame_idx}   GT: {gt_name}   Pred: {pred_name}   {correct}',
            fontsize=9, color=color, fontweight='bold',
        )
    else:
        msg = f'Frame {frame_idx}' + ('\n(video not found)' if vp else '\n(no video_path in h5 attrs)')
        ax.text(0.5, 0.5, msg, ha='center', va='center',
                transform=ax.transAxes, fontsize=11)
        ax.set_title(f'Frame {frame_idx}')

    ax.axis('off')
    plt.tight_layout()
    plt.show()


print('Visualization helpers ready.')


In [ ]:
# ── Interactive viewer ────────────────────────────────
from ipywidgets import Output, VBox, Dropdown, IntSlider
from IPython.display import display, clear_output

chart_out = Output()
frame_out = Output()

# Build display labels: "[fold] filename"
video_options = {
    f'[{fold}] {os.path.basename(vid)}': i
    for i, (fold, vid, _) in enumerate(video_entries)
}

video_dd = Dropdown(
    options=video_options,
    value=0,
    description='Video:',
    style={'description_width': '60px'},
    layout={'width': '700px'},
)

frame_slider = IntSlider(
    min=0, max=0, value=0,
    description='Frame:',
    style={'description_width': '60px'},
    layout={'width': '700px'},
    continuous_update=False,
)


def _refresh(vid_idx, frame_idx):
    with frame_out:
        clear_output(wait=True)
        _draw_frame(frame_idx)
    with chart_out:
        clear_output(wait=True)
        _draw_chart(frame_cursor=frame_idx)


def on_video_change(change):
    vid_idx = change['new']
    _ensure_predictions(vid_idx)
    T = len(_pred_cache['gt'])
    frame_slider.unobserve(on_frame_change, names='value')
    frame_slider.max = T - 1
    frame_slider.value = 0
    frame_slider.observe(on_frame_change, names='value')
    _refresh(vid_idx, 0)


def on_frame_change(change):
    if _pred_cache['vid_idx'] is None:
        return
    _refresh(_pred_cache['vid_idx'], change['new'])


video_dd.observe(on_video_change, names='value')
frame_slider.observe(on_frame_change, names='value')

# Bootstrap: load first video
_ensure_predictions(0)
frame_slider.max = len(_pred_cache['gt']) - 1
_refresh(0, 0)

# frame_out sits between the controls and the chart so it's always visible
display(VBox([video_dd, frame_slider, frame_out, chart_out]))
